# No-crop feature extraction (TCGA and PAIP)

Each non-white 512x512 patch is read **once**, and the **whole** patch is given to every selected model.
One feature vector `(D,)` is saved per patch and model.

### The rule this notebook follows
Every model is defined **once** (section 2): how it is loaded, its transform, and its precision, all copied
from your five-crop notebooks. Both the five-crop path and the no-crop path use that same definition.
The only difference is one line in section 3:

```
five-crop : FiveCrop(256) -> transform each 256x256 crop -> model     (what the old runs did)
no-crop   :                  transform the whole 512x512 -> model     (this notebook)
```

Section 4 **proves** this: it rebuilds stored five-crop features with this notebook's code and compares them
with the files already on disk. If they match, the model, transform and precision are the same as before.

| model | copied from | transform | precision | output |
|---|---|---|---|---|
| `uni2` | `Feature_Extraction_Uni2h_fivecrop.ipynb` | timm transform from the model config | fp32, `no_grad` | 1536 |
| `h_optimus_1` | `Feature_Extraction_h-optimus-1_crops.ipynb` | `Resize((224,224))`, H-optimus mean/std | fp16 autocast | 1536 |
| `conch_v1` | `Feature_Extraction_ConchV1_fivecrop.ipynb` | CONCH preprocess (448) | fp32, `no_grad` | 512 |
| `conch_v1_5` | `Feature_Extraction_Conch_fivecrop.ipynb` | TITAN `return_conch()` transform (448) | fp32, `no_grad` | 768 |
| `virchow2` | `Feature_Extraction_Missing_PAIP_fivecrop.ipynb` | `Resize((224,224))`, model mean/std | fp16 autocast | 2560 |

PRISM is not here: it needs its own environment.

### Run order
1. Restart the kernel and run sections 1-3.
2. Run the checks in section 4. Do not start the full runs unless they all print `PASS`.
3. Run the cohort you want in section 6. One cohort at a time.

## 1. Settings
Everything you might need to change is in this cell.

In [ ]:
import os
# Gated models load from the local Hugging Face cache. Must be set before timm / transformers are imported.
os.environ["HF_HUB_OFFLINE"] = "1"

import sys
import random
from collections import deque
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from pathlib import Path
from typing import Callable

import pandas as pd
import torch
from PIL import Image
from torchvision import transforms
from tqdm import tqdm

# ---- Paths -------------------------------------------------------------------
BASE = Path(r"D:\Aamir Gulzar\KSA_project2")
REPO = BASE / "Cancer-detection-classifier"
sys.path.insert(0, str(REPO / "feature_extraction"))      # repo-local `conch` package
CONCH_V1_CKPT = REPO / "feature_extraction" / "checkpoints" / "CONCH" / "pytorch_model.bin"

# Each cohort folder holds: patch_data/<slide>/<patch>.png, final_merged_without_white.csv, Features/
COHORTS = {
    "tcga": BASE / "dataset",
    "paip": BASE / "paip_data",
}

# Where the new no-crop features go (inside <cohort>/Features/)
NOCROP_FOLDERS = {
    "uni2":        "UNI2H_nocrop",
    "h_optimus_1": "H_Optimus_1_nocrop",
    "conch_v1":    "Conch_v1_nocrop",
    "conch_v1_5":  "Conch_v1_5_nocrop",
    "virchow2":    "Virchow2_nocrop",
}

# Existing five-crop features (inside <cohort>/Features/). Only the checks in section 4 read these.
# The folder names differ between cohorts.
FIVECROP_FOLDERS = {
    "paip": {
        "uni2":        "Uni_2h_fivecrop_Updated",
        "h_optimus_1": "H_Optimus_1_fivecrop_Updated",
        "conch_v1":    "Conch_v1_fivecrop_Updated",
        "conch_v1_5":  "Conch_v1_5_fivecrop_Updated",
        "virchow2":    "Virchow2_fivecrop_Updated",
    },
    "tcga": {
        "uni2":        "Uni_2h_fivecrop_Updated",
        "h_optimus_1": "H_Optimus_1_fivecrop_Updated",
        "conch_v1":    "Conch_v1_fivecrop",
        "conch_v1_5":  "Conch_v1_5_fivecrop_Updated",   # mixed: some slides used 224-px crops, others 256
        "virchow2":    "Virchow2_fivecrop",
    },
}

# ---- Speed -------------------------------------------------------------------
BATCH_SIZE  = 64   # patches per GPU call
NUM_WORKERS = 16   # threads reading + decoding PNGs

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# The five-crop runs used PyTorch defaults. Leave these as they are: turning TF32 on would change fp32 results.
print("device              :", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")
print("TF32 matmul (False) :", torch.backends.cuda.matmul.allow_tf32)

## 2. Models
Each model is described by two things, both copied from its five-crop notebook:

- **`transform`**: PIL image -> tensor. The exact transform the five-crop run applied to each crop.
- **`embed`**: batch of tensors -> features `(B, D)`. The exact model call and precision of the five-crop run.

Two precision modes are used, matching the five-crop runs:
- `fp32`: plain `torch.no_grad()` (UNI2, CONCH v1, CONCH v1.5)
- `fp16`: `torch.autocast(float16)` + `torch.inference_mode()` (H-optimus-1, Virchow2)

Features are returned and saved as float32, like the five-crop files.

In [ ]:
@dataclass
class Model:
    name: str
    dim: int
    transform: Callable   # PIL image -> (3, H, W)
    embed: Callable       # (B, 3, H, W) -> (B, dim) float32 on CPU


def run_fp32(fn, x):
    """Same as the five-crop runs of UNI2, CONCH v1, CONCH v1.5: plain no_grad, fp32."""
    with torch.no_grad():
        out = fn(x.to(DEVICE))
    return out.float().cpu()


def run_fp16(fn, x, after=None):
    """Same as the five-crop runs of H-optimus-1 and Virchow2: fp16 autocast + inference_mode.
    `after` runs outside the autocast block but still in inference_mode, as in the Virchow2 original."""
    with torch.inference_mode():
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16):
            out = fn(x.to(DEVICE))
        if after is not None:
            out = after(out)
    return out.float().cpu()


def load_uni2():
    # From UNI2ExtractorFiveCrop
    import timm
    from timm.data import resolve_data_config
    from timm.data.transforms_factory import create_transform

    net = timm.create_model(
        "hf-hub:MahmoodLab/UNI2-h", pretrained=True,
        img_size=224, patch_size=14, depth=24, num_heads=24, init_values=1e-5, embed_dim=1536,
        mlp_ratio=2.66667 * 2, num_classes=0, no_embed_class=True,
        mlp_layer=timm.layers.SwiGLUPacked, act_layer=torch.nn.SiLU, reg_tokens=8, dynamic_img_size=True,
    ).to(DEVICE).eval()
    transform = create_transform(**resolve_data_config(net.pretrained_cfg, model=net))
    return Model("uni2", 1536, transform, lambda x: run_fp32(net, x))


def load_h_optimus_1():
    # From HOptimusExtractorCrops
    import timm

    net = timm.create_model("hf-hub:bioptimus/H-optimus-1", pretrained=True,
                            init_values=1e-5, dynamic_img_size=False).to(DEVICE).eval()
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.707223, 0.578729, 0.703617), std=(0.211883, 0.230117, 0.177517)),
    ])
    return Model("h_optimus_1", 1536, transform, lambda x: run_fp16(net, x))


def load_conch_v1():
    # From ConchFeatureExtractorFiveCrop (CONCH v1, local checkpoint)
    from conch.open_clip_custom import create_model_from_pretrained

    net, preprocess = create_model_from_pretrained("conch_ViT-B-16", str(CONCH_V1_CKPT))
    net = net.to(DEVICE).eval()
    encode = lambda x: net.encode_image(x, proj_contrast=False, normalize=False)
    return Model("conch_v1", 512, preprocess, lambda x: run_fp32(encode, x))


def load_conch_v1_5():
    # From ConchFeatureExtractorFiveCrop (CONCH v1.5 inside TITAN)
    from transformers import AutoModel

    titan = AutoModel.from_pretrained("MahmoodLab/TITAN", trust_remote_code=True)
    net, transform = titan.return_conch()
    del titan
    net = net.to(DEVICE).eval()
    return Model("conch_v1_5", 768, transform, lambda x: run_fp32(net, x))


def load_virchow2():
    # From Virchow2FiveCrop
    import timm
    from timm.data import resolve_data_config
    from timm.layers import SwiGLUPacked

    net = timm.create_model("hf-hub:paige-ai/Virchow2", pretrained=True,
                            mlp_layer=SwiGLUPacked, act_layer=torch.nn.SiLU).to(DEVICE).eval()
    cfg = resolve_data_config(net.pretrained_cfg, model=net)
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=cfg["mean"], std=cfg["std"]),
    ])

    def tokens_to_vector(out):                                   # out: (B, 261, 1280)
        cls_token = out[:, 0]                                    # (B, 1280)
        patch_mean = out[:, 5:].mean(dim=1)                      # (B, 1280); tokens 1-4 are registers
        return torch.cat([cls_token, patch_mean], dim=-1)        # (B, 2560)

    # As in Virchow2FiveCrop: model inside autocast, token pooling just after it
    return Model("virchow2", 2560, transform, lambda x: run_fp16(net, x, after=tokens_to_vector))


LOADERS = {
    "uni2": load_uni2,
    "h_optimus_1": load_h_optimus_1,
    "conch_v1": load_conch_v1,
    "conch_v1_5": load_conch_v1_5,
    "virchow2": load_virchow2,
}

_loaded = {}   # models stay in GPU memory between cells


def get_model(name):
    if name not in _loaded:
        print(f"loading {name} ...", flush=True)
        _loaded[name] = LOADERS[name]()
    return _loaded[name]

## 3. The only difference: crop or no crop
Both functions use the same `model.transform`. The five-crop version is only used by the checks.

In [ ]:
FIVECROP = transforms.FiveCrop(256)


def prepare_nocrop(model, img):
    """NEW: the whole 512x512 patch -> model.transform -> (3, H, W)"""
    return model.transform(img)


def prepare_fivecrop(model, img):
    """OLD five-crop runs: five 256x256 crops -> model.transform each -> (5, 3, H, W)"""
    return torch.stack([model.transform(crop) for crop in FIVECROP(img)])


def open_patch(path):
    return Image.open(path).convert("RGB")   # same as the five-crop runs

## 4. Checks (run these before the full run)

| check | what it proves | pass if |
|---|---|---|
| **A** | Rebuilds stored **five-crop** features with this notebook's code. Model, transform and precision are the same as before. | cosine >= 0.9999 |
| **B** | Batching 64 patches together gives the same features as one at a time. | cosine >= 0.9999 |
| **C** | TCGA only: matches the **no-crop** UNI2 and H-optimus-1 features that already exist. | cosine >= 0.9999 |

A small difference in the last decimal places is normal: the GPU can add numbers in a slightly different
order when the batch size changes. A cosine of 0.9999 or more means the features are the same for any classifier.

In [ ]:
PASS_COSINE = 0.9999


def sample_patches(cohort, slide, n, seed=0):
    """n random non-white patch files from one slide."""
    root = COHORTS[cohort]
    valid = set(pd.read_csv(root / "final_merged_without_white.csv", usecols=["patch_name"])["patch_name"].astype(str))
    pngs = sorted(f.name for f in os.scandir(root / "patch_data" / slide) if f.name in valid)
    random.Random(seed).shuffle(pngs)
    return [root / "patch_data" / slide / p for p in pngs[:n]]


def compare(new, old, label):
    """Compare two feature tensors of the same shape, row by row."""
    new, old = new.float().reshape(-1, new.shape[-1]), old.float().reshape(-1, old.shape[-1])
    cos = torch.nn.functional.cosine_similarity(new, old, dim=1)
    ok = cos.min().item() >= PASS_COSINE
    print(f"  {label:28s} min cosine {cos.min().item():.7f}   max abs diff {(new - old).abs().max().item():.5f}"
          f"   {'PASS' if ok else 'FAIL'}")
    return ok


def check_A_fivecrop(cohort, models, slide, n=32):
    """Recompute stored five-crop features, one patch (5 crops) per call like the original runs."""
    print(f"Check A, {cohort}, slide {slide}, {n} patches")
    for name in models:
        model = get_model(name)
        folder = COHORTS[cohort] / "Features" / FIVECROP_FOLDERS[cohort][name] / slide
        new, old = [], []
        for path in sample_patches(cohort, slide, n):
            stored = folder / (path.stem + ".pt")
            if not stored.exists():
                continue
            new.append(model.embed(prepare_fivecrop(model, open_patch(path))))
            old.append(torch.load(stored, map_location="cpu"))
        if not old:
            print(f"  {name:28s} SKIPPED - NOT TESTED: no stored five-crop files in {folder}")
            continue
        compare(torch.stack(new), torch.stack(old), f"{name} ({len(old)} patches)")


def check_B_batching(cohort, models, slide, n=64):
    """No-crop features: one batch of n vs one patch at a time."""
    print(f"Check B, {cohort}, slide {slide}, {n} patches")
    paths = sample_patches(cohort, slide, n)
    for name in models:
        model = get_model(name)
        x = torch.stack([prepare_nocrop(model, open_patch(p)) for p in paths])
        batched = model.embed(x)
        single = torch.cat([model.embed(x[i:i + 1]) for i in range(len(x))])
        compare(batched, single, name)


def check_C_existing_nocrop(cohort, models, slide, n=64):
    """No-crop features vs no-crop files already on disk (TCGA UNI2 and H-optimus-1)."""
    print(f"Check C, {cohort}, slide {slide}, {n} patches")
    paths = sample_patches(cohort, slide, n)
    for name in models:
        model = get_model(name)
        folder = COHORTS[cohort] / "Features" / NOCROP_FOLDERS[name] / slide
        keep = [p for p in paths if (folder / (p.stem + ".pt")).exists()]
        if not keep:
            print(f"  {name:28s} SKIPPED - NOT TESTED: no existing no-crop files in {folder}")
            continue
        new = model.embed(torch.stack([prepare_nocrop(model, open_patch(p)) for p in keep]))
        old = torch.stack([torch.load(folder / (p.stem + ".pt"), map_location="cpu") for p in keep])
        compare(new, old, f"{name} ({len(keep)} patches)")

In [ ]:
ALL_MODELS = list(LOADERS)

check_A_fivecrop("paip", ALL_MODELS, slide="training_data_01_MSIH")
check_A_fivecrop("tcga", ALL_MODELS, slide="TCGA-EI-6882_MSIH")
# TCGA CONCH v1.5 five-crop is mixed (224- and 256-px crops by slide), so it can FAIL on some slides.
# That is a problem with the old five-crop features, not with this notebook.

In [ ]:
check_B_batching("paip", ALL_MODELS, slide="training_data_01_MSIH")

In [ ]:
check_C_existing_nocrop("tcga", ["uni2", "h_optimus_1"], slide="TCGA-EI-6882_MSIH")

## 5. The extraction pipeline
How it works:
1. **List jobs**: every non-white patch that at least one selected model has not saved yet.
2. **Read**: `NUM_WORKERS` threads open each PNG once and run every model's transform on it, staying ahead of the GPU.
3. **GPU**: for each batch of `BATCH_SIZE` patches, each model runs in turn.
4. **Save**: two background threads write one `.pt` per patch and model, through a `.tmp` file and a rename,
   so a crash never leaves a half-written file. Rerunning continues where it stopped.

In [ ]:
@dataclass
class Job:
    png: Path
    save_to: dict   # model name -> .pt path, only for models that still need this patch


def list_jobs(cohort, models, slides=None, out_root=None, read_only=False):
    """read_only=True only counts; it creates no folders and deletes nothing."""
    root = COHORTS[cohort]
    out_root = Path(out_root) if out_root else root / "Features"
    valid = set(pd.read_csv(root / "final_merged_without_white.csv", usecols=["patch_name"])["patch_name"].astype(str))

    all_slides = sorted(d.name for d in os.scandir(root / "patch_data") if d.is_dir())
    if slides is not None:
        unknown = set(slides) - set(all_slides)
        if unknown:
            raise ValueError(f"unknown slide(s): {sorted(unknown)}")
        all_slides = [s for s in all_slides if s in set(slides)]

    jobs = []
    done = {m: 0 for m in models}
    total = 0
    for slide in tqdm(all_slides, desc="Listing slides", unit="slide"):
        pngs = sorted(f.name for f in os.scandir(root / "patch_data" / slide) if f.name in valid)
        total += len(pngs)

        existing = {}
        for m in models:
            folder = out_root / NOCROP_FOLDERS[m] / slide
            if not read_only:
                folder.mkdir(parents=True, exist_ok=True)
                for f in folder.glob("*.tmp"):    # leftovers from an interrupted run
                    f.unlink()
            existing[m] = ({f.name[:-3] for f in os.scandir(folder) if f.name.endswith(".pt")}
                           if folder.is_dir() else set())

        for png in pngs:
            stem = png[:-4]
            save_to = {}
            for m in models:
                if stem in existing[m]:
                    done[m] += 1
                else:
                    save_to[m] = out_root / NOCROP_FOLDERS[m] / slide / f"{stem}.pt"
            if save_to:
                jobs.append(Job(root / "patch_data" / slide / png, save_to))

    print(f"{cohort}: {total:,} non-white patches in {len(all_slides)} slides")
    for m in models:
        print(f"  {m:12s} {done[m]:>10,} done   {total - done[m]:>10,} to do   -> {out_root / NOCROP_FOLDERS[m]}")
    return jobs


def read_job(job, models):
    """Runs in a reader thread: open the PNG once, transform it for every model that needs it."""
    img = open_patch(job.png)
    return {m: prepare_nocrop(models[m], img) for m in job.save_to}


def prefetch(fn, items, pool, ahead):
    """Yield (item, result, error) in order, with `ahead` reads running in the background."""
    items = iter(items)
    window = deque()
    for _ in range(ahead):
        item = next(items, None)
        if item is None:
            break
        window.append((item, pool.submit(fn, item)))
    while window:
        item, fut = window.popleft()
        nxt = next(items, None)
        if nxt is not None:
            window.append((nxt, pool.submit(fn, nxt)))
        try:
            yield item, fut.result(), None
        except Exception as e:
            yield item, None, e


def embed_safe(model, x):
    """Run a batch; if the GPU runs out of memory, split it in half and retry."""
    try:
        return model.embed(x)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if len(x) == 1:
            raise
        tqdm.write(f"  [out of memory] {model.name}: splitting batch of {len(x)}")
        half = len(x) // 2
        return torch.cat([embed_safe(model, x[:half]), embed_safe(model, x[half:])])


def save_atomic(tensor, path):
    tmp = str(path) + ".tmp"
    torch.save(tensor, tmp)
    os.replace(tmp, path)


class Saver:
    """Background .pt writer that reports failures."""

    def __init__(self, threads=2, max_pending=5000):
        self.pool = ThreadPoolExecutor(threads)
        self.pending = deque()
        self.max_pending = max_pending
        self.failed = {}

    def save(self, model_name, tensor, path):
        # clone(): a row of the batch tensor would otherwise save the whole batch
        self.pending.append((model_name, path, self.pool.submit(save_atomic, tensor.clone(), path)))
        self._collect(self.max_pending)

    def _collect(self, keep):
        while self.pending and (self.pending[0][2].done() or len(self.pending) > keep):
            m, path, fut = self.pending.popleft()
            try:
                fut.result()
            except Exception as e:
                self.failed[m] = self.failed.get(m, 0) + 1
                tqdm.write(f"  [save failed] {m} {path}: {e}")

    def close(self):
        self._collect(0)
        self.pool.shutdown()


def run_nocrop(cohort, models, slides=None, out_root=None, check_only=False):
    """Extract no-crop features for `models` on `cohort`. Safe to stop and rerun."""
    models = list(LOADERS) if models == "all" else list(models)
    jobs = list_jobs(cohort, models, slides, out_root, read_only=check_only)
    if check_only or not jobs:
        return

    loaded = {m: get_model(m) for m in models if any(m in j.save_to for j in jobs)}
    if DEVICE.type == "cuda":
        print(f"models on GPU: {torch.cuda.memory_allocated() / 2**30:.1f} GB")

    failed = {m: 0 for m in loaded}
    saver = Saver()
    pbar = tqdm(total=len(jobs), desc="Extracting " + ", ".join(loaded), unit="patch")

    with ThreadPoolExecutor(NUM_WORKERS) as readers:
        batch = []
        stream = prefetch(lambda j: read_job(j, loaded), jobs, readers, ahead=BATCH_SIZE * 4)

        for i, (job, tensors, err) in enumerate(stream, 1):
            if err is not None:
                tqdm.write(f"  [read failed] {job.png}: {err}")
                for m in job.save_to:
                    failed[m] += 1
                pbar.update(1)
            else:
                batch.append((job, tensors))

            if len(batch) == BATCH_SIZE or (i == len(jobs) and batch):
                for m, model in loaded.items():
                    rows = [(job, t[m]) for job, t in batch if m in job.save_to]
                    if not rows:
                        continue
                    try:
                        feats = embed_safe(model, torch.stack([t for _, t in rows]))
                    except Exception as e:
                        tqdm.write(f"  [model failed] {m}, {len(rows)} patches: {e}")
                        failed[m] += len(rows)
                        continue
                    for (job, _), f in zip(rows, feats):
                        saver.save(m, f, job.save_to[m])
                pbar.update(len(batch))
                batch = []

    saver.close()
    pbar.close()
    for m in loaded:
        n_fail = failed[m] + saver.failed.get(m, 0)
        n_all = sum(m in j.save_to for j in jobs)
        print(f"  {m:12s} {n_all - n_fail:,} saved, {n_fail:,} failed")
    if any(failed.values()) or saver.failed:
        print("Run the same cell again to retry the failed patches.")

## 6. Run

**Progress check (no GPU, safe any time, writes nothing)**

In [ ]:
run_nocrop("paip", "all", check_only=True)
run_nocrop("tcga", ["conch_v1", "conch_v1_5", "virchow2"], check_only=True)

**Test on one slide** into a separate `_nocrop_test` folder, so the real folders stay untouched. Delete it afterwards.

In [ ]:
run_nocrop("paip", "all", slides=["training_data_46_nonMSIH"],
           out_root=COHORTS["paip"] / "Features" / "_nocrop_test")

**PAIP, all five models** (about 3.5 h)

In [ ]:
run_nocrop("paip", "all")

**TCGA, three models** (about 10 h). UNI2 and H-optimus-1 no-crop already exist for TCGA.

In [ ]:
run_nocrop("tcga", ["conch_v1", "conch_v1_5", "virchow2"])